# 🧩 Notebook 01 — Module Creation Notebook


This notebook creates and tests all core Python modules in the `src/` directory:
- `data.py` – Data loading, cleaning, and splitting  
- `features.py` – Preprocessing pipeline (scaling, encoding, imputation)  
- `train.py` – Model training, cross-validation, and saving artifacts  
- `evaluate.py` – Model evaluation and visualization

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

#Example: point to your project folder in Drive
project_src_path = '/content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction/src'

if os.path.exists(project_src_path):
    os.chdir(project_src_path)
    print(f"Changed directory to: {os.getcwd()}")
else:
    print(f"Error: Directory not found at {project_src_path}")
    print("Please ensure Google Drive is mounted and the path is correct.")

Mounted at /content/drive
Changed directory to: /content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction/src


In [ ]:
%%writefile setup_dependencies.py


"""

setup_dependencies.py
--------------------

A reusable utility module for installing dependencies and generating a requirements.txt file
for reproducibility ML experiments


Usage:
    python src/setup_dependencies.py
"""



import subprocess
import sys
import pathlib import Path



#Depencies list
DEPENDENCIES = [
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "xgboost",
    "jupyter",
    "streamlit"
]


REQUIREMENTS_CONTENT = """\
pandas>=1.5.0
numpy>=1.21.0
scikit-learn>=1.0.0
matplotlib>=3.5.0
seaborn>=0.11.0
xgboost>=1.5.0
streamlit>=1.12.0
"""



def install_dependencies():
  """
  Install all listed dependencies using pip.
  """
  print("## 📦 Installing Dependencies ##")
  for package in DEPENDENCIES:
       print(f"Installing: {package} ...")
       subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
  print("✅ All dependencies installed successfully.\n")


def create_requirements_txt(output_path: str = "requirements.txt"):
    """

    Creates (or overwrites) a requirements.txt file based with pinned versions.
    """
    req_path = Path(output_path)
    req_path.write_text(REQUIREMENTS_CONTENT.strip())
    print(f"📝 requirements.txt created at: {req_path.resolve()}\n")


def set_environment():
  """

  Installs dependencies and generates a requirements.txt file with pinned versions.
  """

  install_dependencies()
  create_requirements_txt()


#Allow direct execution
if __name__ == "__main__":
    setup_environment()

Overwriting setup_dependencies.py


In [ ]:
%%writefile data.py


"""
src/data.py
-----------
Handles loading, cleaning, and splitting the Heart Disease dataset
without internet download dependency.
"""

import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths
RAW_CSV = Path("data/raw/data.csv")
PROC_DIR = Path("data/processed")

# Expected columns for validation
EXPECTED_COLS = {
    "age","sex","cp","trestbps","chol","fbs","restecg",
    "thalach","exang","oldpeak","slope","ca","thal","target"
}

# -----------------------------------------------------------
# Helper function for headerless UCI datasets
# -----------------------------------------------------------
def fix_headerless_uci_dataset(csv_path="data/raw/data.csv"):
    """
    Fixes a UCI Heart Disease dataset that has no headers
    and multi-class 'num' column (0–4) as target.
    Converts 'num' into binary 'target' and saves it back.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"File not found: {csv_path}")

    # Read without header
    df = pd.read_csv(csv_path, header=None)

    # Assign proper columns
    df.columns = [
        "age","sex","cp","trestbps","chol","fbs","restecg",
        "thalach","exang","oldpeak","slope","ca","thal","num"
    ]

    # Convert num→target and drop num
    df["target"] = (df["num"] > 0).astype(int)
    df.drop(columns=["num"], inplace=True)

    # Save back
    df.to_csv(csv_path, index=False)
    print(f"✅ Fixed headerless dataset saved to: {csv_path}")
    return df


# -----------------------------------------------------------
# Main function to load dataset
# -----------------------------------------------------------
def load_data() -> pd.DataFrame:
    """
    Loads and validates the Heart Disease dataset.
    Automatically fixes headerless version if detected.
    """
    if not RAW_CSV.exists():
        raise FileNotFoundError(
            f"❌ Dataset not found at {RAW_CSV}. "
            "Please upload heart.csv to 'data/raw/'."
        )

    # Try reading normally
    try:
        df = pd.read_csv(RAW_CSV)
    except Exception:
        df = pd.read_csv(RAW_CSV, header=None)

    # Detect headerless dataset (columns are numeric)
    if all(str(c).isdigit() for c in df.columns):
        print("⚠️ Detected headerless dataset — applying fix.")
        df = fix_headerless_uci_dataset(str(RAW_CSV))

    # Verify correct columns
    missing = EXPECTED_COLS.difference(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    # Clean duplicates and shuffle
    df = df.drop_duplicates().sample(frac=1, random_state=42).reset_index(drop=True)

    print("✅ Dataset loaded successfully.")
    return df


# -----------------------------------------------------------
# Split data into train/validation/test
# -----------------------------------------------------------
def split_save(df: pd.DataFrame, test_size=0.2, val_size=0.2, seed=42):
    """
    Split the dataset into train, validation, and test sets
    and save them to 'data/processed/'.
    """
    X = df.drop(columns=["target"])
    y = df["target"]

    # Split train+val vs test
    X_trv, X_test, y_trv, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )

    # Split train vs val
    X_train, X_val, y_train, y_val = train_test_split(
        X_trv, y_trv, test_size=val_size, stratify=y_trv, random_state=seed
    )

    # Save splits
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    X_train.to_csv(PROC_DIR / "X_train.csv", index=False)
    X_val.to_csv(PROC_DIR / "X_val.csv", index=False)
    X_test.to_csv(PROC_DIR / "X_test.csv", index=False)
    y_train.to_csv(PROC_DIR / "y_train.csv", index=False)
    y_val.to_csv(PROC_DIR / "y_val.csv", index=False)
    y_test.to_csv(PROC_DIR / "y_test.csv", index=False)

    print("✅ Train/Validation/Test splits saved in data/processed/")


if __name__ == "__main__":
    df = load_data()
    split_save(df)

Overwriting data.py


In [ ]:
%%writefile features.py
"""
features.py
------------
Feature engineering module for Heart Disease Prediction.
Contains reusable functions for derived feature creation,
feature preprocessing, and feature name extraction.
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer, FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer


# ----------------------------------------------------------------
# 1️⃣ Derived Feature Creation
# ----------------------------------------------------------------
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add clinically meaningful derived features.

    hr_reserve  = thalach / (220 - age)
    bp_per_age  = trestbps / age
    chol_per_age = chol / age
    age_bin     = 5 quantile bins of age
    """
    out = df.copy()

    # Avoid division by zero
    out["age"] = out["age"].clip(lower=1)
    out["trestbps"] = out["trestbps"].clip(lower=1)
    out["chol"] = out["chol"].clip(lower=1)
    out["thalach"] = out["thalach"].clip(lower=1)

    # 1️⃣ Heart Rate Reserve
    max_hr_pred = 220 - out["age"]
    out["hr_reserve"] = (out["thalach"] / max_hr_pred).clip(0, 3)

    # 2️⃣ BP per Age
    out["bp_per_age"] = (out["trestbps"] / out["age"]).clip(0, 5)

    # 3️⃣ Chol per Age
    out["chol_per_age"] = (out["chol"] / out["age"]).clip(0, 10)

    # 4️⃣ Age Bin (quantile-based)
    kbinner = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
    out["age_bin"] = kbinner.fit_transform(out[["age"]]).astype(int)

    return out


# ----------------------------------------------------------------
# 2️⃣ Feature Name Extraction Helper
# ----------------------------------------------------------------
def get_feature_names(preproc: Pipeline, num_cols: list[str], cat_cols: list[str],
                      engineered_num: list[str], engineered_cat: list[str]) -> list[str]:
    """
    Extracts human-readable feature names from a fitted preprocessing pipeline.
    Works with numeric + one-hot encoded categorical features.
    """
    derived_numeric = num_cols + engineered_num
    derived_categor = cat_cols + engineered_cat

    # Numeric pass-through features
    num_names = derived_numeric

    # Get one-hot feature names for categorical
    ohe = preproc.named_steps["cols"].named_transformers_["cat"].named_steps["onehot"]
    ohe_out = list(ohe.get_feature_names_out(derived_categor))

    return num_names + ohe_out


# ----------------------------------------------------------------
# 3️⃣ Preprocessor Builder (optional utility)
# ----------------------------------------------------------------
def build_preprocessor(num_cols, cat_cols):
    """
    Build the full preprocessing pipeline with numeric/categorical transformations
    and derived feature generation.
    """
    ENGINEERED_NUM = ["hr_reserve", "bp_per_age", "chol_per_age"]
    ENGINEERED_CAT = ["age_bin"]

    num_cols_all = num_cols + ENGINEERED_NUM
    cat_cols_all = cat_cols + ENGINEERED_CAT

    num_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    cat_pipe = make_pipeline(SimpleImputer(strategy="most_frequent"),
                             OneHotEncoder(handle_unknown="ignore", sparse_output=False))

    deriver = FunctionTransformer(add_derived_features, feature_names_out="one-to-one", validate=False)

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols_all),
        ("cat", cat_pipe, cat_cols_all)
    ])

    full_pipeline = Pipeline([
        ("derive", deriver),
        ("cols", preprocessor)
    ])

    return full_pipeline


Overwriting features.py


In [ ]:
%%writefile train.py
# src/train.py
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.data import PROC_DIR, TARGET, NUM_COLS, CAT_COLS  # for consistency
from src.features import build_preprocessor

MODELS_DIR = Path("models")
REPORTS_DIR = Path("reports")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def _load_train_val():
    X_train = pd.read_csv(PROC_DIR / "X_train.csv")
    y_train = pd.read_csv(PROC_DIR / "y_train.csv").squeeze("columns")
    X_val = pd.read_csv(PROC_DIR / "X_val.csv")
    y_val = pd.read_csv(PROC_DIR / "y_val.csv").squeeze("columns")
    return X_train, y_train, X_val, y_val

def _pick_model(name: str):
    if name == "logreg":
        return LogisticRegression(max_iter=200, class_weight="balanced")
    if name == "rf":
        return RandomForestClassifier(
            n_estimators=400, random_state=42, class_weight="balanced"
        )
    raise ValueError("Unknown model. Use 'logreg' or 'rf'.")

def train(model: str = "logreg") -> str:
    X_train, y_train, X_val, y_val = _load_train_val()
    pre = build_preprocessor()
    clf = _pick_model(model)

    pipe = Pipeline([("pre", pre), ("model", clf)])

    # Cross-validated ROC-AUC on train
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"CV ROC-AUC ({model}): {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

    # Fit & validate
    pipe.fit(X_train, y_train)
    val_proba = pipe.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_proba)
    print(f"Validation ROC-AUC: {val_auc:.3f}")

    run_id = f"{model}_seed42"
    joblib.dump(pipe, MODELS_DIR / f"heart_model_{run_id}.pkl")
    (REPORTS_DIR / f"metrics_{run_id}.json").write_text(
        json.dumps(
            {"cv_auc_mean": float(cv_auc.mean()),
             "cv_auc_std": float(cv_auc.std()),
             "val_auc": float(val_auc)},
            indent=2
        )
    )
    print(f"💾 Saved: models/heart_model_{run_id}.pkl")
    print(f"📝 Saved: reports/metrics_{run_id}.json")
    return run_id

if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["logreg", "rf"], default="logreg")
    args = ap.parse_args()
    train(args.model)


Overwriting train.py


In [ ]:
%%writefile evaluate.py
"""
evaluate.py
------------
Evaluate trained models for Heart Disease Prediction.
Generates ROC, Precision-Recall, Confusion Matrix, and Calibration plots.

Usage (from terminal or Colab):
    python src/evaluate.py --run_id logreg_seed42
"""

import json
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
)
from sklearn.calibration import CalibrationDisplay

# Directories
MODELS_DIR = Path("models")
REPORTS_DIR = Path("reports")
PROC_DIR = Path("data/processed")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
#  Helper functions
# ---------------------------------------------------------------------

def load_splits():
    """Load test split for evaluation."""
    X_test = pd.read_csv(PROC_DIR / "X_test.csv")
    y_test = pd.read_csv(PROC_DIR / "y_test.csv").squeeze()
    return X_test, y_test


def evaluate(run_id: str):
    """
    Evaluate a trained model using stored pickle artifact.
    Args:
        run_id: Model identifier (e.g., 'logreg_seed42' or 'rf_seed42').
    """
    model_path = MODELS_DIR / f"heart_model_{run_id}.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found at {model_path}")

    # Load model and test data
    model = joblib.load(model_path)
    X_test, y_test = load_splits()

    # Predict probabilities and labels
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)

    # -----------------------------------------------------------------
    #  Metrics
    # -----------------------------------------------------------------
    roc_auc = auc(*roc_curve(y_test, proba)[:2][::-1])
    pr_auc = average_precision_score(y_test, proba)
    cm = confusion_matrix(y_test, preds)
    report = classification_report(y_test, preds, output_dict=True)

    metrics_dict = {
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "accuracy": float(report["accuracy"]),
        "precision": float(report["1"]["precision"]),
        "recall": float(report["1"]["recall"]),
        "f1_score": float(report["1"]["f1-score"]),
    }

    # Save metrics
    metrics_path = REPORTS_DIR / f"test_metrics_{run_id}.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics_dict, f, indent=2)
    print(f"✅ Metrics saved to {metrics_path}")

    # -----------------------------------------------------------------
    #  Plots
    # -----------------------------------------------------------------
    # ROC
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.figure()
    RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc).plot()
    plt.title(f"ROC Curve (AUC={roc_auc:.3f})")
    plt.savefig(REPORTS_DIR / f"roc_curve_{run_id}.png", bbox_inches="tight")
    plt.close()

    # Precision-Recall
    precision, recall, _ = precision_recall_curve(y_test, proba)
    plt.figure()
    plt.plot(recall, precision, label=f"PR-AUC={pr_auc:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.savefig(REPORTS_DIR / f"pr_curve_{run_id}.png", bbox_inches="tight")
    plt.close()

    # Confusion Matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")
    plt.title("Confusion Matrix (threshold=0.5)")
    plt.savefig(REPORTS_DIR / f"confusion_matrix_{run_id}.png", bbox_inches="tight")
    plt.close()

    # Calibration Curve
    plt.figure()
    CalibrationDisplay.from_predictions(y_test, proba, n_bins=10)
    plt.title("Calibration Curve")
    plt.savefig(REPORTS_DIR / f"calibration_curve_{run_id}.png", bbox_inches="tight")
    plt.close()

    print("📊 Evaluation plots saved in 'reports/' folder.")

    # -----------------------------------------------------------------
    #  Summary Output
    # -----------------------------------------------------------------
    print("=== Evaluation Summary ===")
    for k, v in metrics_dict.items():
        print(f"{k:<15}: {v:.3f}")

    return metrics_dict


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Evaluate trained Heart Disease model.")
    parser.add_argument("--run_id", required=True, help="Run ID, e.g., logreg_seed42")
    args = parser.parse_args()

    evaluate(args.run_id)


Overwriting evaluate.py
